In [24]:
import pandas as pd
import numpy as np
import re

### Load Data

In [25]:
df = pd.read_csv("../data/Raw_ice_data.csv")
 
print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

/var/folders/q0/89dmr2hd7dj4vckq7lmx8ltw0000gn/T/ipykernel_9133/2311061698.py:1: DtypeWarning: Columns (20,39,40,41,42,43,44,45,46,47,50,51,52,53,54,55,57,58,59,60,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Raw_ice_data.csv")


Original shape: (750000, 65)
Columns: ['Stay Book In Date', 'Detention Book In Date', 'Detention Facility', 'Detention Facility Code', 'Detention Book Out Date', 'Detention Release Reason', 'Stay Book Out Date', 'Stay Release Reason', 'Religion', 'Marital', 'Gender', 'Birth Date', 'Ethnicity', 'Birth Year', 'Entry Status', 'Bond Posted Date', 'Bond Posted Amount', 'Initial Bond Set Amount', 'Case Status', 'Case Category', 'Final Order Yes No', 'Final Order Date', 'Departed Date', 'Departure Country', 'Case Threat Level', 'Charge', 'Anonymized Identifier', 'days_to_stay', 'days_to_removal', 'running_arrest_count', 'running_detention_count', 'running_removal_count', 'stay_inauguration_year', 'arrests_anchor_stay_date', 'arrests_Apprehension Date', 'arrests_Apprehension Method', 'arrests_Arrest Created By', 'arrests_TOA Current Duty AOR', 'arrests_Anonymized Identifier', 'removals_anchor_stay_date', 'removals_Case Category', 'removals_Final Order Yes No', 'removals_Final Order Date', 'rem

### DROP REDUNDANT / DUPLICATE COLUMNS
- The removals_* nested columns largely duplicate top-level detention columns or have very high missingness due to FOIA redactions.
- We keep top-level columns as the primary source of truth.

In [26]:
cols_to_drop = [
    # Duplicate identifiers
    'arrests_Anonymized Identifier',
    'removals_Anonymized Identifier',
    'arrests_Arrest Created By',          # Redacted (b)(6)(b)(7)(c) throughout
 
    # Anchor dates — redundant with Stay Book In Date
    'arrests_anchor_stay_date',
    'removals_anchor_stay_date',
 
    # Removals columns that duplicate top-level or are heavily redacted
    'removals_Final Order Yes No',        # Redundant with top-level Final Order Yes No
    'removals_Final Order Date',          # Redundant with top-level Final Order Date
    'removals_Gender',                    # Redundant with top-level Gender
    'removals_Departure Country',         # Redundant with top-level Departure Country
    'removals_Case Status',               # Redundant with top-level Case Status
    'removals_Case Category',             # Redundant with top-level Case Category
    'removals_Birth Date',                # High FOIA redaction
    'removals_Birth Year',                # Redundant with top-level Birth Year
    'removals_Entry Status',              # Redundant with top-level Entry Status
    'removals_Apprehension Date',         # Redundant with arrests_Apprehension Date
    'removals_Case Threat Level',         # Redundant with top-level Case Threat Level
    'removals_Birth Country',             # Very high missingness
    'removals_Citizenship Country',       # Very high missingness
    'removals_Entry Date',                # Very high missingness
    'removals_MSC Charge Date',           # Very high missingness
    'removals_MSC Conviction Date',       # Very high missingness
    'removals_Port of Departure',         # Very high missingness
    'removals_Departure Date',            # Redundant with top-level Departed Date
    'removals_Anonymized Identifier',     # Already listed above (safety)
 
    # Top-level columns not useful for modeling
    'Anonymized Identifier',              # Non-informative ID
    'Birth Date',                         # Replaced by Birth Year
    'Detention Facility Code',            # Redundant with Detention Facility name

    # Leaky post-outcome dates
    'Final Order Date',
    'Departed Date',
    'Departure Country',

    # Redundant with year
    'stay_inauguration_year',
]
 
# Only drop columns that actually exist
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)
 
print(f"\nShape after dropping redundant columns: {df.shape}")


Shape after dropping redundant columns: (750000, 35)


### DROP ROWS WHERE TARGET VARIABLE IS NULL

In [27]:
df = df.dropna(subset=['Final Order Yes No'])
print(f"Shape after dropping null target rows: {df.shape}")
 
# Create binary numeric target
df['final_order_numeric'] = df['Final Order Yes No'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df = df.dropna(subset=['final_order_numeric'])
df['final_order_numeric'] = df['final_order_numeric'].astype(int)
 
# Drop the raw version
df = df.drop(columns=['Final Order Yes No'])

print(f"Target distribution:\n{df['final_order_numeric'].value_counts()}")
 

Shape after dropping null target rows: (749495, 35)
Target distribution:
final_order_numeric
1    458769
0    290726
Name: count, dtype: int64


### HANDLE FOIA REDACTIONS

In [28]:
# Values like (b)(6)(b)(7)(c) are privacy redactions — treat as missing
 
foia_pattern = r'\(b\)\(\d+\)'
obj_cols = df.select_dtypes(include='object').columns
 
print("\nFOIA redaction cleaning:")
for col in obj_cols:
    col_as_str = df[col].astype(str)
    mask = col_as_str.str.contains(foia_pattern, regex=True, na=False)
    if mask.sum() > 0:
        df.loc[mask, col] = np.nan
        print(f"  Replaced {mask.sum():,} redacted values in: {col}")
 
print(f"\nShape after FOIA cleaning: {df.shape}")


FOIA redaction cleaning:

Shape after FOIA cleaning: (749495, 35)


### PARSE DATE COLUMNS

In [29]:
date_cols = [
    'Stay Book In Date',
    'Detention Book In Date',
    'Detention Book Out Date',
    'Stay Book Out Date',
    'Final Order Date',
    'Departed Date',
    'Bond Posted Date',
    'arrests_Apprehension Date',
    'removals_MSC Charge Date',
    'removals_Entry Date',
]
 
date_cols = [c for c in date_cols if c in df.columns]
 
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

### DERIVE YEAR + ADMINISTRATION

In [30]:
# Use Stay Book In Date as the primary event year
df['year'] = df['Stay Book In Date'].dt.year
 
# Fall back to arrests_Apprehension Date if Stay Book In Date is missing
if 'arrests_Apprehension Date' in df.columns:
    df['year'] = df['year'].fillna(df['arrests_Apprehension Date'].dt.year)
 
# Fall back to stay_inauguration_year if both are missing
if 'stay_inauguration_year' in df.columns:
    df['year'] = df['year'].fillna(df['stay_inauguration_year'])
 
df['year'] = df['year'].astype('Int64')
 
def get_administration(year):
    if pd.isna(year):
        return 'Unknown'
    year = int(year)
    if 2009 <= year <= 2016:
        return 'Obama'
    elif 2017 <= year <= 2020:
        return 'Trump_1'
    elif 2021 <= year <= 2024:
        return 'Biden'
    elif year >= 2025:
        return 'Trump_2'
    else:
        return 'Unknown'
 
df['administration'] = df['year'].apply(get_administration)
 
print("\nAdministration distribution:")
print(df['administration'].value_counts())


Administration distribution:
administration
Obama      247555
Trump_1    200234
Biden      199695
Trump_2    102007
Unknown         4
Name: count, dtype: int64


### NUMERIC COLUMNS + FILL IN MISSING

In [31]:
numeric_cols = [
    'Birth Year',
    'Bond Posted Amount',
    'Initial Bond Set Amount',
    'days_to_stay',
    'days_to_removal',
    'running_arrest_count',
    'running_detention_count',
    'running_removal_count',
]
numeric_cols = [c for c in numeric_cols if c in df.columns]
 
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
 
# Fill with median
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

### CATEGORICAL COLUMNS — STANDARDIZE + FILL MISSING

In [32]:
categorical_cols = [
    'Gender',
    'Ethnicity',
    'Marital',
    'Religion',
    'Entry Status',
    'Case Status',
    'Case Category',
    'Case Threat Level',
    'Detention Release Reason',
    'Stay Release Reason',
    'Departure Country',
    'Detention Facility',
    'Charge',
    'arrests_Apprehension Method',
    'removals_MSC Charge',
    'removals_MSC Charge Code',
    'removals_MSC Criminal Charge Status',
    'removals_Processing Disposition',
    'removals_Current Program',
    'removals_Charge Code',
]
categorical_cols = [c for c in categorical_cols if c in df.columns]
 
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace({'Nan': 'Unknown', 'None': 'Unknown', 'Na': 'Unknown', '': 'Unknown'})
    df[col] = df[col].fillna('Unknown')

### FEATURE ENGINEERING

In [33]:
# --- Age at detention ---
df['age_at_detention'] = df['year'].astype('float') - df['Birth Year']
df['age_at_detention'] = df['age_at_detention'].apply(
    lambda x: x if 0 < x < 120 else np.nan
)
 
# --- Detention duration (days) ---
if 'Detention Book In Date' in df.columns and 'Detention Book Out Date' in df.columns:
    df['detention_duration_days'] = (
        df['Detention Book Out Date'] - df['Detention Book In Date']
    ).dt.days
    # Negative or extreme values are likely errors
    df['detention_duration_days'] = df['detention_duration_days'].apply(
        lambda x: x if (x is not None and 0 <= x <= 3650) else np.nan
    )
 
# --- Stay extended beyond detention ---
if 'Stay Book Out Date' in df.columns and 'Detention Book Out Date' in df.columns:
    df['stay_extended'] = (
        df['Stay Book Out Date'] > df['Detention Book Out Date']
    ).astype(int)
 
# --- Arrest method flags ---
if 'arrests_Apprehension Method' in df.columns:
    df['is_CAP_arrest'] = df['arrests_Apprehension Method'].str.contains('Cap', na=False).astype(int)
    df['is_287g_arrest'] = df['arrests_Apprehension Method'].str.contains('287', na=False).astype(int)
    df['is_custodial_arrest'] = df['arrests_Apprehension Method'].str.contains('Custodial', na=False).astype(int)
    df['is_border_arrest'] = df['arrests_Apprehension Method'].str.contains('Border', na=False).astype(int)
 
# --- Demographic flags ---
df['is_hispanic'] = (df['Ethnicity'] == 'Hispanic Origin').astype(int)
df['is_male'] = (df['Gender'] == 'Male').astype(int)
 
# --- Entry status flags ---
df['is_PWA'] = df['Entry Status'].str.contains('Pwa|Without Authorization', case=False, na=False).astype(int)
df['is_asylum_seeker'] = df['Entry Status'].str.contains('Asylum|Refugee', case=False, na=False).astype(int)
 
# --- Criminal charge flags ---
if 'removals_MSC Charge' in df.columns:
    df['has_criminal_charge'] = (df['removals_MSC Charge'] != 'Unknown').astype(int)
 
if 'removals_MSC Criminal Charge Status' in df.columns:
    df['is_convicted'] = df['removals_MSC Criminal Charge Status'].str.contains(
        'Convict', case=False, na=False
    ).astype(int)
 
# --- Bond flag ---
if 'Bond Posted Amount' in df.columns:
    df['bond_posted'] = (df['Bond Posted Amount'] > 0).astype(int)
 
# --- Final order issued before departure ---
if 'Final Order Date' in df.columns and 'Departed Date' in df.columns:
    df['order_before_departure'] = (
        df['Final Order Date'] < df['Departed Date']
    ).astype(int)
 
# --- Juvenile flag ---
df['is_juvenile'] = (df['age_at_detention'] < 18).astype(int)
 
print(f"\nFeature engineering complete. Shape: {df.shape}")


Feature engineering complete. Shape: (749495, 52)


### FINAL MISSING VALUE CHECK

In [34]:
print("\nMissing values in remaining columns:")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
 
missing_pct = (missing / len(df) * 100).round(2)
print("\nMissing %:")
print(missing_pct)


Missing values in remaining columns:
Bond Posted Date                660690
arrests_TOA Current Duty AOR    637872
arrests_Apprehension Date       451833
Stay Book Out Date               30148
Detention Book Out Date          14976
detention_duration_days          14976
age_at_detention                   498
dtype: int64

Missing %:
Bond Posted Date                88.15
arrests_TOA Current Duty AOR    85.11
arrests_Apprehension Date       60.28
Stay Book Out Date               4.02
Detention Book Out Date          2.00
detention_duration_days          2.00
age_at_detention                 0.07
dtype: float64


### DROP COLUMNS WITH >60% MISSINGNESS

In [35]:
threshold = 0.6
miss_rate = df.isna().mean()
high_miss = miss_rate[miss_rate > threshold].index.tolist()
 
print(f"\nDropping {len(high_miss)} columns with >60% missingness:")
print(high_miss)
 
df = df.drop(columns=high_miss)
print(f"Shape after dropping high-missingness columns: {df.shape}")


Dropping 3 columns with >60% missingness:
['Bond Posted Date', 'arrests_Apprehension Date', 'arrests_TOA Current Duty AOR']
Shape after dropping high-missingness columns: (749495, 49)


### DROP LOW-VALUE AND LEAKY COLUMNS POST FEATURE ENGINEERING

In [36]:
cols_to_drop_post = [
    # Leaky — derived from outcome-adjacent dates, not available pre-outcome
    'stay_extended',
    'order_before_departure',

    # Religion — too inconsistent to clean reliably (mixed codes, typos, NTA errors)
    'Religion',

    # Bond — only ~9% of rows have values, too sparse to be reliable
    'Bond Posted Amount',
    'Initial Bond Set Amount',
    'bond_posted',

    # Criminal charge raw columns — replaced by charge_category and has_criminal_charge
    'removals_MSC Charge Code',   # code version of above, redundant
    'removals_Charge Code',       # redundant with charge_category
    'Charge',                     # top-level version, redundant with charge_category
]

cols_to_drop_post = [c for c in cols_to_drop_post if c in df.columns]
df = df.drop(columns=cols_to_drop_post)
print(f"Shape after dropping low-value columns: {df.shape}")

Shape after dropping low-value columns: (749495, 41)


### FINAL DATASET SUMMARY

In [37]:
print("\n" + "="*60)
print("FINAL CLEANED DATASET SUMMARY")
print("="*60)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution (final_order_numeric):")
print(df['final_order_numeric'].value_counts())
print(f"\nAdministration breakdown:")
print(df['administration'].value_counts())
print(f"\nColumn list:")
print(df.columns.tolist())


FINAL CLEANED DATASET SUMMARY
Shape: (749495, 41)

Target distribution (final_order_numeric):
final_order_numeric
1    458769
0    290726
Name: count, dtype: int64

Administration breakdown:
administration
Obama      247555
Trump_1    200234
Biden      199695
Trump_2    102007
Unknown         4
Name: count, dtype: int64

Column list:
['Stay Book In Date', 'Detention Book In Date', 'Detention Facility', 'Detention Book Out Date', 'Detention Release Reason', 'Stay Book Out Date', 'Stay Release Reason', 'Marital', 'Gender', 'Ethnicity', 'Birth Year', 'Entry Status', 'Case Status', 'Case Category', 'Case Threat Level', 'days_to_stay', 'days_to_removal', 'running_arrest_count', 'running_detention_count', 'running_removal_count', 'arrests_Apprehension Method', 'removals_MSC Charge', 'removals_Processing Disposition', 'removals_MSC Criminal Charge Status', 'removals_Current Program', 'final_order_numeric', 'year', 'administration', 'age_at_detention', 'detention_duration_days', 'is_CAP_arres

### SAVE CLEANED DATASET

In [38]:
df.to_csv("../data/new_ICE_Master_Cleaned.csv", index=False)
print("\nSaved to new_ICE_Master_Cleaned.csv")


Saved to new_ICE_Master_Cleaned.csv


In [39]:
print(df.columns.tolist())


['Stay Book In Date', 'Detention Book In Date', 'Detention Facility', 'Detention Book Out Date', 'Detention Release Reason', 'Stay Book Out Date', 'Stay Release Reason', 'Marital', 'Gender', 'Ethnicity', 'Birth Year', 'Entry Status', 'Case Status', 'Case Category', 'Case Threat Level', 'days_to_stay', 'days_to_removal', 'running_arrest_count', 'running_detention_count', 'running_removal_count', 'arrests_Apprehension Method', 'removals_MSC Charge', 'removals_Processing Disposition', 'removals_MSC Criminal Charge Status', 'removals_Current Program', 'final_order_numeric', 'year', 'administration', 'age_at_detention', 'detention_duration_days', 'is_CAP_arrest', 'is_287g_arrest', 'is_custodial_arrest', 'is_border_arrest', 'is_hispanic', 'is_male', 'is_PWA', 'is_asylum_seeker', 'has_criminal_charge', 'is_convicted', 'is_juvenile']
